# Single-subject sEEG + motion + events review

This notebook inspects one subject/session/run and ties together:

- EDF iEEG/sEEG data
- iEEG channel metadata
- BIDS motion data
- motion channel metadata
- behavioral events
- recording timing

It is intended to live in:

`code/ipynb/inspect_seeg_subject.ipynb`

Important BIDS detail: `*_motion.tsv` is intentionally **headerless**. Its columns are named using the rows of the corresponding `*_channels.tsv`.

In [ ]:
# Imports

from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import mne
except ImportError as exc:
    raise ImportError(
        "MNE is required to read EDF files. Install it with: pip install mne"
    ) from exc

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("MNE version:", mne.__version__)

In [ ]:
# Configuration

SUBJECT = "sub-BG0001"
SESSION = "ses-01"

# None = automatically choose the first EDF run.
RUN = None

# Window shown in neural and motion plots.
PLOT_START_S = 0
PLOT_DURATION_S = 5

# Number of neural channels to show.
N_NEURAL_CHANNELS = 12

# Plotting only: cap points per trace to keep figures readable and responsive.
MAX_PLOT_POINTS_PER_TRACE = 5000

# Event markers can obscure continuous data if events.tsv contains many
# BCI2000 state changes. Markers are drawn only if the number of unique
# onsets in the displayed window is at or below this value.
SHOW_EVENT_MARKERS = True
MAX_EVENT_MARKERS = 50

# Optional explicit motion channel names.
# Leave None for automatic joystick/cursor/target-first selection.
MOTION_COLUMNS = None
MAX_MOTION_CHANNELS = 8

In [ ]:
# Locate the BIDS dataset root.

def find_dataset_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "dataset_description.json").exists()
            and (candidate / "participants.tsv").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the BIDS dataset root. "
        "Launch Jupyter from somewhere inside the dataset."
    )

DATASET_ROOT = find_dataset_root(Path.cwd())
SESSION_DIR = DATASET_ROOT / SUBJECT / SESSION

print("Dataset root :", DATASET_ROOT)
print("Subject      :", SUBJECT)
print("Session      :", SESSION)
print("Session path :", SESSION_DIR)

if not SESSION_DIR.exists():
    raise FileNotFoundError(f"Session directory does not exist: {SESSION_DIR}")

In [ ]:
# Participant metadata

participants = pd.read_csv(DATASET_ROOT / "participants.tsv", sep="\t", dtype=str)

if "participant_id" in participants.columns:
    participant_row = participants.loc[participants["participant_id"] == SUBJECT]
    if len(participant_row):
        display(participant_row)
    else:
        print(f"{SUBJECT} is not present in participants.tsv")
else:
    print("participants.tsv has no participant_id column.")

In [ ]:
# Discover EDF runs.

ieeg_dir = SESSION_DIR / "ieeg"
edf_files = sorted(ieeg_dir.glob("*_ieeg.edf")) if ieeg_dir.exists() else []

if not edf_files:
    raise FileNotFoundError(f"No *_ieeg.edf files found in {ieeg_dir}")

def extract_run_entity(path: Path):
    match = re.search(r"_run-([^_]+)", path.name)
    return match.group(1) if match else None

edf_by_run = {
    extract_run_entity(path): path
    for path in edf_files
    if extract_run_entity(path) is not None
}

print("Available EDF runs:")
for run_id, path in edf_by_run.items():
    print(f"  run-{run_id}: {path.name}")

if RUN is None:
    RUN = next(iter(edf_by_run))
    print(f"\nRUN was None, so run-{RUN} was selected automatically.")
elif RUN not in edf_by_run:
    raise ValueError(
        f"Requested run-{RUN} not found. Available runs: {list(edf_by_run)}"
    )

EDF_PATH = edf_by_run[RUN]
print("\nSelected EDF:", EDF_PATH)

In [ ]:
# Match files belonging to the selected run.

def first_or_none(paths):
    paths = sorted(paths)
    return paths[0] if paths else None

# iEEG channel metadata
ieeg_channels_path = first_or_none(
    ieeg_dir.glob(f"*run-{RUN}_channels.tsv")
)

# Events may live in beh/ or beside another modality.
events_path = first_or_none(
    SESSION_DIR.rglob(f"*run-{RUN}_events.tsv")
)

# Motion data
motion_dir = SESSION_DIR / "motion"
motion_path = (
    first_or_none(motion_dir.glob(f"*run-{RUN}*_motion.tsv"))
    if motion_dir.exists()
    else None
)

# BIDS motion metadata files have the same prefix as the motion TSV.
motion_channels_path = None
motion_json_path = None

if motion_path is not None:
    motion_channels_path = Path(
        str(motion_path).replace("_motion.tsv", "_channels.tsv")
    )
    motion_json_path = Path(
        str(motion_path).replace("_motion.tsv", "_motion.json")
    )

    if not motion_channels_path.exists():
        motion_channels_path = None
    if not motion_json_path.exists():
        motion_json_path = None

availability = pd.DataFrame({
    "data": [
        "EDF neural data",
        "iEEG channels.tsv",
        "events.tsv",
        "motion.tsv",
        "motion channels.tsv",
        "motion.json",
    ],
    "found": [
        EDF_PATH.exists(),
        ieeg_channels_path is not None,
        events_path is not None,
        motion_path is not None,
        motion_channels_path is not None,
        motion_json_path is not None,
    ],
    "path": [
        str(EDF_PATH),
        str(ieeg_channels_path) if ieeg_channels_path else None,
        str(events_path) if events_path else None,
        str(motion_path) if motion_path else None,
        str(motion_channels_path) if motion_channels_path else None,
        str(motion_json_path) if motion_json_path else None,
    ],
})

display(availability)

In [ ]:
# Inspect iEEG channel metadata.

ieeg_channels = None

if ieeg_channels_path is not None:
    ieeg_channels = pd.read_csv(ieeg_channels_path, sep="\t")
    print("iEEG channels file:", ieeg_channels_path.name)
    print("Shape:", ieeg_channels.shape)
    display(ieeg_channels.head(25))

    if "type" in ieeg_channels.columns:
        print("\nChannel types:")
        display(
            ieeg_channels["type"]
            .value_counts(dropna=False)
            .rename("count")
            .to_frame()
        )
else:
    print("No matching iEEG channels.tsv found.")

In [ ]:
# Load and inspect events.

events = None

if events_path is not None:
    events = pd.read_csv(events_path, sep="\t")
    print("Events file:", events_path.name)
    print("Shape:", events.shape)
    print("Columns:", list(events.columns))
    display(events.head(25))

    if "onset" in events.columns:
        onset_numeric = pd.to_numeric(events["onset"], errors="coerce")
        print("\nValid event rows :", int(onset_numeric.notna().sum()))
        print("Unique onsets    :", int(onset_numeric.nunique(dropna=True)))

    # Show useful low-cardinality categorical columns.
    for col in events.columns:
        nunique = events[col].nunique(dropna=True)
        if 1 < nunique <= 20:
            print(f"\n{col} counts:")
            display(
                events[col]
                .value_counts(dropna=False)
                .head(20)
                .rename("count")
                .to_frame()
            )
else:
    print("No matching events.tsv found.")

In [ ]:
# Event timeline.
#
# If there are extremely many events, plotting each event individually is not
# useful. In that case this cell reports the density instead of making a solid plot.

if events is None:
    print("Event timeline skipped: no events.tsv.")
elif "onset" not in events.columns:
    print("Event timeline skipped: events.tsv has no onset column.")
else:
    ev = events.copy()
    ev["onset"] = pd.to_numeric(ev["onset"], errors="coerce")
    ev = ev.dropna(subset=["onset"])

    if len(ev) > 5000:
        print(
            f"events.tsv contains {len(ev):,} rows. "
            "Skipping the full event scatter because it would be visually saturated."
        )

        counts, edges = np.histogram(ev["onset"], bins=100)

        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(edges[:-1], counts, linewidth=1)
        ax.set_xlabel("Time from recording start (s)")
        ax.set_ylabel("Events per time bin")
        ax.set_title(f"{SUBJECT} {SESSION} run-{RUN}: event density")
        plt.tight_layout()
        plt.show()
    else:
        label_col = next(
            (
                c for c in [
                    "trial_type",
                    "event_type",
                    "condition",
                    "value",
                    "TaskCondition",
                    "TaskBlockType",
                ]
                if c in ev.columns
            ),
            None,
        )

        if label_col is None:
            ev["_event"] = "event"
            label_col = "_event"

        labels = ev[label_col].astype(str)
        categories = list(pd.unique(labels))
        category_to_y = {label: i for i, label in enumerate(categories)}

        fig, ax = plt.subplots(
            figsize=(14, max(3, min(9, 0.4 * len(categories) + 2)))
        )
        ax.scatter(
            ev["onset"],
            labels.map(category_to_y),
            s=10,
            alpha=0.7,
        )
        ax.set_yticks(range(len(categories)))
        ax.set_yticklabels(categories)
        ax.set_xlabel("Time from recording start (s)")
        ax.set_ylabel(label_col)
        ax.set_title(f"{SUBJECT} {SESSION} run-{RUN}: event timeline")
        plt.tight_layout()
        plt.show()

In [ ]:
# Load EDF neural data.

raw = mne.io.read_raw_edf(
    EDF_PATH,
    preload=False,
    infer_types=True,
    verbose="ERROR",
)

print(raw)
print(f"\nSampling frequency : {raw.info['sfreq']:.3f} Hz")
print(f"Duration           : {raw.times[-1]:.3f} s")
print(f"Number of channels : {len(raw.ch_names)}")

raw_channel_table = pd.DataFrame({
    "name": raw.ch_names,
    "mne_type": raw.get_channel_types(),
})

display(raw_channel_table.head(30))

In [ ]:
# Choose actual neural channels for plotting.
#
# If channels.tsv contains BIDS channel types, prefer SEEG/ECOG/DBS/EEG and
# exclude obvious reference/non-neural channels. Otherwise fall back to the
# first channels in the EDF.

NEURAL_BIDS_TYPES = {"SEEG", "ECOG", "DBS", "EEG"}

plot_channel_names = []

if (
    ieeg_channels is not None
    and "name" in ieeg_channels.columns
    and "type" in ieeg_channels.columns
):
    channel_type_lookup = {
        str(row["name"]): str(row["type"]).upper()
        for _, row in ieeg_channels.iterrows()
    }

    plot_channel_names = [
        ch
        for ch in raw.ch_names
        if channel_type_lookup.get(ch, "") in NEURAL_BIDS_TYPES
    ]

if not plot_channel_names:
    plot_channel_names = [
        ch for ch in raw.ch_names
        if not re.search(r"(^REF|REFERENCE|TRIG|STATUS)", ch, re.IGNORECASE)
    ]

plot_channel_names = plot_channel_names[:N_NEURAL_CHANNELS]

if not plot_channel_names:
    plot_channel_names = raw.ch_names[:N_NEURAL_CHANNELS]

print("Neural channels selected:")
for ch in plot_channel_names:
    print(" ", ch)

In [ ]:
# Prepare sparse event markers for continuous-data plots.

def event_markers_for_window(events, start_s, stop_s):
    if (
        not SHOW_EVENT_MARKERS
        or events is None
        or "onset" not in events.columns
    ):
        return np.array([])

    onsets = pd.to_numeric(events["onset"], errors="coerce").dropna().to_numpy()
    onsets = onsets[(onsets >= start_s) & (onsets <= stop_s)]
    onsets = np.unique(onsets)

    if len(onsets) > MAX_EVENT_MARKERS:
        print(
            f"Not overlaying event markers: {len(onsets)} unique event onsets "
            f"fall inside this window, above MAX_EVENT_MARKERS={MAX_EVENT_MARKERS}."
        )
        return np.array([])

    return onsets

In [ ]:
# Plot EDF neural data.
#
# This is a static overview. It:
#   - limits the displayed window
#   - uses only selected neural channels
#   - downsamples for DISPLAY ONLY if needed
#   - scales each trace robustly so one artifact does not obscure all channels
#
# The raw EDF itself is never modified.

sfreq = float(raw.info["sfreq"])
start_s = float(PLOT_START_S)
stop_s = min(start_s + float(PLOT_DURATION_S), float(raw.times[-1]))

start_sample = max(0, int(round(start_s * sfreq)))
stop_sample = min(raw.n_times, int(round(stop_s * sfreq)))

if stop_sample <= start_sample:
    raise ValueError("Requested neural plot window is empty.")

picks = [raw.ch_names.index(ch) for ch in plot_channel_names]
data, times = raw[picks, start_sample:stop_sample]

# MNE returns voltage channels in SI volts. Convert to µV for display.
data_uv = data * 1e6
data_uv = data_uv - np.nanmedian(data_uv, axis=1, keepdims=True)

# Downsample only the plotted arrays, never the source data.
plot_step = max(1, int(np.ceil(data_uv.shape[1] / MAX_PLOT_POINTS_PER_TRACE)))
times_plot = times[::plot_step]
data_plot = data_uv[:, ::plot_step]

# Robust per-channel normalization for a readable stacked overview.
# The scale factor for every channel is printed below.
scales = np.nanpercentile(np.abs(data_plot), 95, axis=1)
scales[~np.isfinite(scales) | (scales == 0)] = 1.0

normalized = data_plot / scales[:, None]
normalized = np.clip(normalized, -4, 4)

spacing = 10.0
offsets = np.arange(len(picks))[::-1] * spacing

fig, ax = plt.subplots(
    figsize=(16, max(6, 0.55 * len(picks) + 2))
)

for i, ch in enumerate(plot_channel_names):
    ax.plot(
        times_plot,
        normalized[i] + offsets[i],
        linewidth=0.6,
    )

for onset in event_markers_for_window(events, start_s, stop_s):
    ax.axvline(onset, linewidth=0.8, alpha=0.25)

ax.set_yticks(offsets)
ax.set_yticklabels(plot_channel_names)
ax.set_xlabel("Time from recording start (s)")
ax.set_ylabel("Channel")
ax.set_title(
    f"{SUBJECT} {SESSION} run-{RUN}: EDF neural data "
    f"({start_s:.1f}-{stop_s:.1f} s)"
)
ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

scale_table = pd.DataFrame({
    "channel": plot_channel_names,
    "95th_percentile_abs_amplitude_uV": scales,
})

print(
    f"Displayed every {plot_step} sample(s) for plotting only. "
    "Each channel is individually normalized in this overview."
)
display(scale_table)

In [ ]:
# Optional MNE browser.
#
# This uses MNE's own continuous-signal visualization and automatic display
# scaling. In some VS Code/Jupyter setups it will appear inline; in others it
# may open an interactive Matplotlib window.

try:
    mne_fig = raw.plot(
        picks=plot_channel_names,
        start=PLOT_START_S,
        duration=PLOT_DURATION_S,
        n_channels=len(plot_channel_names),
        scalings="auto",
        show_scrollbars=False,
        show_scalebars=True,
        block=False,
        title=f"{SUBJECT} {SESSION} run-{RUN}",
    )
except Exception as exc:
    print("MNE browser could not be displayed in this notebook backend:")
    print(exc)

In [ ]:
# Load BIDS motion data CORRECTLY.
#
# BIDS *_motion.tsv files do not have a header row.
# The corresponding *_channels.tsv gives the channel order and metadata.

motion = None
motion_channels = None
motion_meta = {}

if motion_path is None:
    print("No matching motion.tsv found.")
else:
    # Headerless by BIDS specification.
    motion = pd.read_csv(
        motion_path,
        sep="\t",
        header=None,
        na_values=["n/a"],
    )

    if motion_channels_path is not None:
        motion_channels = pd.read_csv(
            motion_channels_path,
            sep="\t",
            dtype=str,
        )

        if len(motion_channels) != motion.shape[1]:
            warnings.warn(
                "Motion column count does not match the number of rows in "
                "motion channels.tsv. Keeping generic column names."
            )
            motion.columns = [
                f"channel_{i:03d}" for i in range(motion.shape[1])
            ]
        else:
            # Build readable and unique labels.
            labels = []

            for i, row in motion_channels.iterrows():
                name = str(row.get("name", f"channel_{i:03d}"))
                component = str(row.get("component", "n/a"))

                if component.lower() not in {"n/a", "nan", "none", ""}:
                    label = f"{name}_{component}"
                else:
                    label = name

                labels.append(label)

            # Guarantee uniqueness even if metadata labels repeat.
            seen = {}
            unique_labels = []

            for label in labels:
                count = seen.get(label, 0)
                unique_labels.append(
                    label if count == 0 else f"{label}_{count + 1}"
                )
                seen[label] = count + 1

            motion.columns = unique_labels
    else:
        motion.columns = [
            f"channel_{i:03d}" for i in range(motion.shape[1])
        ]

    if motion_json_path is not None:
        with open(motion_json_path, "r", encoding="utf-8") as f:
            motion_meta = json.load(f)

    print("Motion file:", motion_path.name)
    print("Shape:", motion.shape)
    print("Columns:", list(motion.columns))

    display(motion.head(10))

    if motion_channels is not None:
        print("\nMotion channel metadata:")
        display(motion_channels.head(25))

    if motion_meta:
        print("\nMotion sidecar:")
        display(pd.Series(motion_meta, name="value").to_frame())

In [ ]:
# Motion timing.

motion_time = None
motion_sampling_frequency = None

if motion is not None:
    try:
        motion_sampling_frequency = float(
            motion_meta.get("SamplingFrequency")
        )
    except (TypeError, ValueError):
        motion_sampling_frequency = None

    if motion_sampling_frequency is not None:
        motion_time = (
            np.arange(len(motion), dtype=float)
            / motion_sampling_frequency
        )
        print(
            f"Motion sampling frequency: "
            f"{motion_sampling_frequency:g} Hz"
        )
    else:
        motion_time = np.arange(len(motion), dtype=float)
        warnings.warn(
            "SamplingFrequency is missing or invalid in motion.json. "
            "Motion x-axis will use row number instead of seconds."
        )

In [ ]:
# Select motion channels for plotting.

selected_motion_columns = []

if motion is not None:
    if MOTION_COLUMNS is not None:
        selected_motion_columns = [
            c for c in MOTION_COLUMNS if c in motion.columns
        ]

        missing_requested = [
            c for c in MOTION_COLUMNS if c not in motion.columns
        ]

        if missing_requested:
            warnings.warn(
                f"Requested motion channels not found: {missing_requested}"
            )
    else:
        numeric_columns = []

        for col in motion.columns:
            values = pd.to_numeric(motion[col], errors="coerce")
            if values.notna().any():
                numeric_columns.append(col)

        preferred_pattern = re.compile(
            r"(joystick|cursor|target|position|pos|movement)",
            re.IGNORECASE,
        )

        preferred = [
            c for c in numeric_columns
            if preferred_pattern.search(c)
        ]
        remaining = [
            c for c in numeric_columns
            if c not in preferred
        ]

        selected_motion_columns = (
            preferred + remaining
        )[:MAX_MOTION_CHANNELS]

    print("Motion channels selected:")
    for col in selected_motion_columns:
        print(" ", col)

In [ ]:
# Plot motion data.

if motion is None:
    print("Motion plot skipped: motion.tsv missing.")
elif not selected_motion_columns:
    print("Motion plot skipped: no numeric motion channels found.")
else:
    time = np.asarray(motion_time)

    if motion_sampling_frequency is not None:
        window_mask = (
            np.isfinite(time)
            & (time >= start_s)
            & (time <= stop_s)
        )
    else:
        window_mask = np.isfinite(time)

    if not window_mask.any():
        print(
            "No motion samples fall inside the selected window. "
            "Plotting the full motion recording."
        )
        window_mask = np.isfinite(time)

    selected_time = time[window_mask]

    # Keep display responsive for long/high-frequency recordings.
    plot_step_motion = max(
        1,
        int(np.ceil(
            len(selected_time) / MAX_PLOT_POINTS_PER_TRACE
        )),
    )

    n_axes = len(selected_motion_columns)

    fig, axes = plt.subplots(
        n_axes,
        1,
        figsize=(16, max(3, 2.0 * n_axes)),
        sharex=True,
    )

    if n_axes == 1:
        axes = [axes]

    marker_onsets = event_markers_for_window(
        events,
        start_s,
        stop_s,
    )

    for ax, col in zip(axes, selected_motion_columns):
        values = pd.to_numeric(
            motion[col],
            errors="coerce",
        ).to_numpy()

        x = time[window_mask][::plot_step_motion]
        y = values[window_mask][::plot_step_motion]

        ax.plot(x, y, linewidth=0.8)
        ax.set_ylabel(col)
        ax.grid(axis="x", alpha=0.2)

        for onset in marker_onsets:
            ax.axvline(
                onset,
                linewidth=0.8,
                alpha=0.2,
            )

    axes[-1].set_xlabel(
        "Time from recording start (s)"
        if motion_sampling_frequency is not None
        else "Sample row"
    )

    fig.suptitle(
        f"{SUBJECT} {SESSION} run-{RUN}: motion data",
        y=1.0,
    )

    plt.tight_layout()
    plt.show()

    print(
        f"Displayed every {plot_step_motion} motion sample(s) "
        "for plotting only."
    )

In [ ]:
# Compare timing across streams.

summary_rows = [{
    "stream": "EDF neural",
    "rows_or_samples": raw.n_times,
    "sampling_frequency_hz": raw.info["sfreq"],
    "start_s": 0.0,
    "end_s": float(raw.times[-1]),
}]

if motion is not None:
    summary_rows.append({
        "stream": "Motion",
        "rows_or_samples": len(motion),
        "sampling_frequency_hz": motion_sampling_frequency,
        "start_s": 0.0 if motion_sampling_frequency is not None else np.nan,
        "end_s": (
            float(motion_time[-1])
            if motion_sampling_frequency is not None and len(motion_time)
            else np.nan
        ),
    })

if events is not None and "onset" in events.columns:
    event_onsets = pd.to_numeric(
        events["onset"],
        errors="coerce",
    ).dropna()

    summary_rows.append({
        "stream": "Events",
        "rows_or_samples": len(events),
        "sampling_frequency_hz": np.nan,
        "start_s": (
            float(event_onsets.min())
            if len(event_onsets)
            else np.nan
        ),
        "end_s": (
            float(event_onsets.max())
            if len(event_onsets)
            else np.nan
        ),
    })

alignment_summary = pd.DataFrame(summary_rows)
display(alignment_summary)

In [ ]:
# Source BCI2000 files for this subject/session.

source_bci_dir = (
    DATASET_ROOT
    / "sourcedata"
    / SUBJECT
    / SESSION
    / "bci2000"
)

if source_bci_dir.exists():
    source_files = sorted(source_bci_dir.iterdir())

    source_table = pd.DataFrame({
        "name": [p.name for p in source_files],
        "suffix": [p.suffix for p in source_files],
        "size_mb": [
            p.stat().st_size / (1024 ** 2)
            for p in source_files
        ],
    })

    display(source_table)
else:
    print("No sourcedata BCI2000 directory found.")

## What changed from the first version

1. `*_motion.tsv` is now read with `header=None`.
2. Motion channel names come from the corresponding `*_channels.tsv`.
3. Event markers are not drawn when they are dense enough to obscure the signal.
4. The neural overview prefers channels marked as `SEEG`, `ECOG`, `DBS`, or `EEG`.
5. The default view is 5 seconds instead of 20 seconds.
6. Continuous plots are reduced to a reasonable number of display points without modifying the source data.
7. An optional native MNE browser is included for inspecting the EDF with MNE's own scaling.